In [0]:
from pyspark.sql import SparkSession

In [0]:
spark = SparkSession.builder.appName("Linear Regresion").getOrCreate()
spark

In [0]:
df = spark.table('workspace.default.tips')
df.show()

+----------+----+------+------+---+------+----+
|total_bill| tip|   sex|smoker|day|  time|size|
+----------+----+------+------+---+------+----+
|     16.99|1.01|Female|    No|Sun|Dinner|   2|
|     10.34|1.66|  Male|    No|Sun|Dinner|   3|
|     21.01| 3.5|  Male|    No|Sun|Dinner|   3|
|     23.68|3.31|  Male|    No|Sun|Dinner|   2|
|     24.59|3.61|Female|    No|Sun|Dinner|   4|
|     25.29|4.71|  Male|    No|Sun|Dinner|   4|
|      8.77| 2.0|  Male|    No|Sun|Dinner|   2|
|     26.88|3.12|  Male|    No|Sun|Dinner|   4|
|     15.04|1.96|  Male|    No|Sun|Dinner|   2|
|     14.78|3.23|  Male|    No|Sun|Dinner|   2|
|     10.27|1.71|  Male|    No|Sun|Dinner|   2|
|     35.26| 5.0|Female|    No|Sun|Dinner|   4|
|     15.42|1.57|  Male|    No|Sun|Dinner|   2|
|     18.43| 3.0|  Male|    No|Sun|Dinner|   4|
|     14.83|3.02|Female|    No|Sun|Dinner|   2|
|     21.58|3.92|  Male|    No|Sun|Dinner|   2|
|     10.33|1.67|Female|    No|Sun|Dinner|   3|
|     16.29|3.71|  Male|    No|Sun|Dinne

In [0]:
df.printSchema()

root
 |-- total_bill: double (nullable = true)
 |-- tip: double (nullable = true)
 |-- sex: string (nullable = true)
 |-- smoker: string (nullable = true)
 |-- day: string (nullable = true)
 |-- time: string (nullable = true)
 |-- size: integer (nullable = true)



In [0]:
df.columns

['total_bill', 'tip', 'sex', 'smoker', 'day', 'time', 'size']

In [0]:
df.describe().show()

+-------+------------------+------------------+------+------+----+------+------------------+
|summary|        total_bill|               tip|   sex|smoker| day|  time|              size|
+-------+------------------+------------------+------+------+----+------+------------------+
|  count|               244|               244|   244|   244| 244|   244|               244|
|   mean|19.785942622950824|2.9982786885245902|  NULL|  NULL|NULL|  NULL| 2.569672131147541|
| stddev| 8.902411954856856|1.3836381890011826|  NULL|  NULL|NULL|  NULL|0.9510998047322332|
|    min|              3.07|               1.0|Female|    No| Fri|Dinner|                 1|
|    max|             50.81|              10.0|  Male|   Yes|Thur| Lunch|                 6|
+-------+------------------+------------------+------+------+----+------+------------------+



In [0]:
print(df.count())
df = df.dropna(how='any')
print(df.count())

244
244


In [0]:
from pyspark.ml.feature import StringIndexer

indexer = StringIndexer(inputCols=['sex', 'smoker', 'day', 'time'], outputCols=['sex_index', 'smoker_index', 'day_index', 'time_index'])


In [0]:
df = indexer.fit(df).transform(df)
df.show()

+----------+----+------+------+---+------+----+---------+------------+---------+----------+
|total_bill| tip|   sex|smoker|day|  time|size|sex_index|smoker_index|day_index|time_index|
+----------+----+------+------+---+------+----+---------+------------+---------+----------+
|     16.99|1.01|Female|    No|Sun|Dinner|   2|      1.0|         0.0|      1.0|       0.0|
|     10.34|1.66|  Male|    No|Sun|Dinner|   3|      0.0|         0.0|      1.0|       0.0|
|     21.01| 3.5|  Male|    No|Sun|Dinner|   3|      0.0|         0.0|      1.0|       0.0|
|     23.68|3.31|  Male|    No|Sun|Dinner|   2|      0.0|         0.0|      1.0|       0.0|
|     24.59|3.61|Female|    No|Sun|Dinner|   4|      1.0|         0.0|      1.0|       0.0|
|     25.29|4.71|  Male|    No|Sun|Dinner|   4|      0.0|         0.0|      1.0|       0.0|
|      8.77| 2.0|  Male|    No|Sun|Dinner|   2|      0.0|         0.0|      1.0|       0.0|
|     26.88|3.12|  Male|    No|Sun|Dinner|   4|      0.0|         0.0|      1.0|

In [0]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(inputCols=['tip', 'size','sex_index', 'smoker_index', 'day_index', 'time_index'], outputCol='feature_vector')
 

In [0]:
df_ = assembler.transform(df)
df_.show()

+----------+----+------+------+---+------+----+---------+------------+---------+----------+--------------------+
|total_bill| tip|   sex|smoker|day|  time|size|sex_index|smoker_index|day_index|time_index|      feature_vector|
+----------+----+------+------+---+------+----+---------+------------+---------+----------+--------------------+
|     16.99|1.01|Female|    No|Sun|Dinner|   2|      1.0|         0.0|      1.0|       0.0|[1.01,2.0,1.0,0.0...|
|     10.34|1.66|  Male|    No|Sun|Dinner|   3|      0.0|         0.0|      1.0|       0.0|[1.66,3.0,0.0,0.0...|
|     21.01| 3.5|  Male|    No|Sun|Dinner|   3|      0.0|         0.0|      1.0|       0.0|[3.5,3.0,0.0,0.0,...|
|     23.68|3.31|  Male|    No|Sun|Dinner|   2|      0.0|         0.0|      1.0|       0.0|[3.31,2.0,0.0,0.0...|
|     24.59|3.61|Female|    No|Sun|Dinner|   4|      1.0|         0.0|      1.0|       0.0|[3.61,4.0,1.0,0.0...|
|     25.29|4.71|  Male|    No|Sun|Dinner|   4|      0.0|         0.0|      1.0|       0.0|[4.71

In [0]:
df_xy = df_.select(['feature_vector','total_bill'])
df_xy.show()

+--------------------+----------+
|      feature_vector|total_bill|
+--------------------+----------+
|[1.01,2.0,1.0,0.0...|     16.99|
|[1.66,3.0,0.0,0.0...|     10.34|
|[3.5,3.0,0.0,0.0,...|     21.01|
|[3.31,2.0,0.0,0.0...|     23.68|
|[3.61,4.0,1.0,0.0...|     24.59|
|[4.71,4.0,0.0,0.0...|     25.29|
|[2.0,2.0,0.0,0.0,...|      8.77|
|[3.12,4.0,0.0,0.0...|     26.88|
|[1.96,2.0,0.0,0.0...|     15.04|
|[3.23,2.0,0.0,0.0...|     14.78|
|[1.71,2.0,0.0,0.0...|     10.27|
|[5.0,4.0,1.0,0.0,...|     35.26|
|[1.57,2.0,0.0,0.0...|     15.42|
|[3.0,4.0,0.0,0.0,...|     18.43|
|[3.02,2.0,1.0,0.0...|     14.83|
|[3.92,2.0,0.0,0.0...|     21.58|
|[1.67,3.0,1.0,0.0...|     10.33|
|[3.71,3.0,0.0,0.0...|     16.29|
|[3.5,3.0,1.0,0.0,...|     16.97|
|(6,[0,1],[3.35,3.0])|     20.65|
+--------------------+----------+
only showing top 20 rows


In [0]:
train_df, test_df = df_xy.dropna().randomSplit([0.75, 0.25])
(train_df.count(), len(train_df.columns)), (test_df.count(), len(test_df.columns))

((183, 2), (61, 2))

In [0]:
from pyspark.ml.regression import LinearRegression

lr = LinearRegression(featuresCol='feature_vector', labelCol='total_bill')
lr = lr.fit(train_df)

In [0]:
lr.coefficients, lr.intercept

(DenseVector([3.2348, 3.7894, -2.502, 2.867, 0.1225, -1.2132]),
 0.4218132991182092)

In [0]:
y_hat = lr.evaluate(test_df)
y_hat.predictions.show()

+--------------------+----------+------------------+
|      feature_vector|total_bill|        prediction|
+--------------------+----------+------------------+
|(6,[0,1],[1.25,2.0])|     10.07|12.044122115791563|
|(6,[0,1],[1.25,2.0])|     10.51|12.044122115791563|
|(6,[0,1],[2.64,3.0])|     17.59|20.329871045023122|
|(6,[0,1],[3.15,3.0])|     20.08| 21.97960072561971|
|(6,[0,1],[3.18,2.0])|     19.82|18.287216789421795|
|(6,[0,1],[6.73,4.0])|     48.27| 37.34948298917781|
|(6,[0,1],[7.58,4.0])|     39.42|40.099032456838785|
| (6,[0,1],[9.0,4.0])|     48.33| 44.69239744987242|
|[1.01,2.0,1.0,0.0...|     16.99| 8.888287709085873|
|[1.25,2.0,1.0,0.0...|      8.51|  8.57402348317543|
|[1.32,2.0,0.0,0.0...|      9.68|12.393099449030409|
|[1.44,2.0,0.0,1.0...|      7.74|15.525727012015224|
|[1.5,2.0,0.0,0.0,...|     19.08|11.884749378343894|
|[1.5,2.0,1.0,0.0,...|     26.41|10.350778260423674|
|[1.5,2.0,1.0,0.0,...|      8.35| 9.382714503075718|
|[1.5,2.0,1.0,0.0,...|     10.65| 9.3827145030

In [0]:
y_hat.meanAbsoluteError, y_hat.meanAbsoluteError, y_hat.r2

(4.874313219190186, 4.874313219190186, 0.5473844630570939)